In [1]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize

def fit_generator(dag, x0, xT, ltt_values, t_values):
    """
    Fit a generator matrix A consistent with DAG, initial/final counts, and LTT values.

    Parameters
    ----------
    dag : (k, k) array
        Adjacency matrix of allowed transitions. dag[i,j] = 1 if i->j allowed, 0 else.
    x0 : (k,) array
        Initial counts per state
    xT : (k,) array
        Final counts per state
    ltt_values : (n,) array
        LTT measurements at times t_values
    t_values : (n,) array
        Time points corresponding to ltt_values

    Returns
    -------
    A_hat : (k, k) array
        Estimated generator matrix
    """
    k = len(x0)
    # Indices of free Q elements according to DAG
    free_q_idx = np.where(dag.T != 0)  # transpose because A = diag + Q^T

    # Initial guess: lambda_i - mu_i = 0.1, Q = 0.1 where allowed
    lambda_mu_guess = np.ones(k) * 0.1
    q_guess = np.ones(len(free_q_idx[0])) * 0.1
    theta0 = np.concatenate([lambda_mu_guess, q_guess])

    # Helper: reconstruct A from parameters
    def build_A(theta):
        lam_mu = theta[:k]
        q_flat = theta[k:]
        Q = np.zeros((k, k))
        Q[free_q_idx] = q_flat
        np.fill_diagonal(Q, -np.sum(Q, axis=1))
        A = np.diag(lam_mu) + Q.T
        return A

    # Loss function: combine endpoint and LTT errors
    def loss(theta):
        A = build_A(theta)
        # endpoint term
        xT_pred = expm(A * t_values[-1]) @ x0
        loss_end = np.sum((xT_pred - xT) ** 2)
        # LTT term
        loss_ltt = 0
        for t_i, L_obs in zip(t_values, ltt_values):
            L_pred = np.sum(expm(A * t_i) @ x0)
            loss_ltt += (L_pred - L_obs) ** 2
        return loss_end + loss_ltt

    # Optimization (bounds: free rates >=0)
    bounds = [(0, None)] * len(theta0)
    res = minimize(loss, theta0, method='L-BFGS-B', bounds=bounds)
    A_hat = build_A(res.x)
    return A_hat

# ======================
# Example usage
# ======================
if __name__ == "__main__":
    # 3-state DAG: 0->1, 0->2, 1->2
    dag = np.array([
        [0,1,1],
        [0,0,1],
        [0,0,0]
    ])
    x0 = np.array([10,0,0])
    xT = np.array([5,3,4])
    t_values = np.linspace(0,5,6)
    # Example LTT values (sum of states over time)
    ltt_values = np.array([10, 9, 8, 7, 6, 5])
    
    A_est = fit_generator(dag, x0, xT, ltt_values, t_values)
    print("Estimated generator A:\n", A_est)

Estimated generator A:
 [[ 0.   0.1  0.1]
 [ 0.   0.   0.1]
 [ 0.   0.  -0.1]]


In [2]:
def decompose_A(A, dag):
    """
    Decompose A into lambda_i - mu_i and Q
    """
    k = A.shape[0]
    Q = np.zeros_like(A)
    # Off-diagonal from A transpose (only where DAG allows)
    for i in range(k):
        for j in range(k):
            if i != j and dag[i,j]:
                Q[i,j] = A[j,i]  # note transpose
    # Diagonal of Q
    for i in range(k):
        Q[i,i] = -np.sum(Q[i,:]) + Q[i,i]  # ensure row sums zero
    # Lambda - Mu from A diagonal minus Q diagonal
    lam_minus_mu = np.diag(A) - np.diag(Q)
    return lam_minus_mu, Q

In [3]:
decompose_A(A_est, dag)

(array([ 0. ,  0. , -0.1]),
 array([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]))

In [4]:
import numpy as np
from scipy.linalg import expm

# DAG: 0->1, 0->2, 1->2
dag = np.array([
    [0,1,1],
    [0,0,1],
    [0,0,0]
])

x0 = np.array([10,0,0])
t_values = np.linspace(0,5,6)

# True generator
lam_minus_mu = np.array([0.2, 0.1, 0.05])
Q_true = np.zeros((3,3))
Q_true[0,1] = 0.05
Q_true[0,2] = 0.03
Q_true[1,2] = 0.02
# Fill diagonals
for i in range(3):
    Q_true[i,i] = -np.sum(Q_true[i,:])

A_true = np.diag(lam_minus_mu) + Q_true.T

# Simulate xT and LTT
X = np.column_stack([expm(A_true*t) @ x0 for t in t_values])
xT = X[:,-1]
ltt_values = X.sum(axis=0)

print("xT =", xT)
print("LTT =", ltt_values)
print("A_true =\n", A_true)

xT = [18.221188    4.12867628  2.49621591]
LTT = [10.         12.1597077  14.66962078 17.5806829  20.9508601  24.8460802 ]
A_true =
 [[0.12 0.   0.  ]
 [0.05 0.08 0.  ]
 [0.03 0.02 0.05]]


In [6]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize

# ---------- Inputs ----------
# DAG: 0->1, 0->2, 1->2
dag = np.array([
    [0,1,1],
    [0,0,1],
    [0,0,0]
])

x0 = np.array([10,0,0])
t_values = np.linspace(0,5,6)

# True generator (for reference)
lam_minus_mu = np.array([0.2, 0.1, 0.05])
Q_true = np.zeros((3,3))
Q_true[0,1] = 0.05
Q_true[0,2] = 0.03
Q_true[1,2] = 0.02
for i in range(3):
    Q_true[i,i] = -np.sum(Q_true[i,:])

A_true = np.diag(lam_minus_mu) + Q_true.T

# Simulate xT and LTT
X = np.column_stack([expm(A_true*t) @ x0 for t in t_values])
xT = X[:,-1]
ltt_values = X.sum(axis=0)

print("xT =", xT)
print("LTT =", ltt_values)
print("A_true =\n", A_true)

# ---------- Step 1: Define optimization ----------
k = len(x0)
free_q_idx = np.where(dag.T != 0)

# FIXED: create initial guess correctly
theta0 = np.concatenate([np.ones(k)*0.1, np.ones(len(free_q_idx[0]))*0.01])

def build_A(theta):
    lam_mu = theta[:k]
    q_flat = theta[k:]
    Q = np.zeros((k,k))
    Q[free_q_idx] = q_flat
    for i in range(k):
        Q[i,i] = -np.sum(Q[i,:])
    A = np.diag(lam_mu) + Q.T
    return A

def loss(theta):
    A = build_A(theta)
    # endpoint loss
    loss_end = np.sum((expm(A*t_values[-1]) @ x0 - xT)**2)
    # LTT loss
    loss_ltt = 0
    for t, L_obs in zip(t_values, ltt_values):
        loss_ltt += (np.sum(expm(A*t) @ x0) - L_obs)**2
    return loss_end + loss_ltt

# ---------- Step 2: Optimize ----------
res = minimize(loss, theta0, method='L-BFGS-B', bounds=[(0,None)]*len(theta0))
A_hat = build_A(res.x)

# ---------- Step 3: Decompose ----------
lam_minus_mu_hat = np.diag(A_hat) - np.diag(np.zeros_like(A_hat))  # only diagonal
Q_hat = np.zeros((k,k))
Q_hat[free_q_idx] = res.x[k:]
for i in range(k):
    Q_hat[i,i] = -np.sum(Q_hat[i,:])

# ---------- Output ----------
print("\nEstimated A:\n", A_hat)
print("\nEstimated lambda - mu:\n", lam_minus_mu_hat)
print("\nEstimated Q:\n", Q_hat)

xT = [18.221188    4.12867628  2.49621591]
LTT = [10.         12.1597077  14.66962078 17.5806829  20.9508601  24.8460802 ]
A_true =
 [[0.12 0.   0.  ]
 [0.05 0.08 0.  ]
 [0.03 0.02 0.05]]

Estimated A:
 [[0.16258173 0.01       0.01      ]
 [0.         0.09       0.01      ]
 [0.         0.         0.08      ]]

Estimated lambda - mu:
 [0.16258173 0.09       0.08      ]

Estimated Q:
 [[-0.    0.    0.  ]
 [ 0.01 -0.01  0.  ]
 [ 0.01  0.01 -0.02]]
